## Loading dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import scipy.stats as stats
folder = '/content/drive/MyDrive/'

Mounted at /content/drive


In [ ]:
df = pd.read_csv(folder + 'bed_based_shelters.csv')
df['OCCUPANCY_DATE'] = pd.to_datetime(df['OCCUPANCY_DATE'], errors='coerce')

In [ ]:
sorted(df['OCCUPANCY_DATE'].dt.year.unique())

[np.int32(2021), np.int32(2022), np.int32(2024), np.int32(2025)]

## Setting up features + splitting

In [ ]:
# get time features set up
df['year'] = df['OCCUPANCY_DATE'].dt.year
df['month'] = df['OCCUPANCY_DATE'].dt.month
df['dayofweek'] = df['OCCUPANCY_DATE'].dt.dayofweek

In [ ]:
# build features + target
df['pressure'] = df['OCCUPIED_BEDS'] / df['CAPACITY_ACTUAL_BED']

# high pressure will be 1 and not high pressure (low pressure/normal) will be 0
# high pressure -> how full the shelther is
df['high_pressure'] = (df['pressure'] > 0.9).astype(int)

# sort data in time orders per shelther
df = df.sort_values(['SHELTER_ID', 'OCCUPANCY_DATE'])

# create the future target (each shelter is treated differently)
# future pressure means the next 2 week's pressure status... we are doing lagged target forecasting
df['future_pressure'] = df.groupby('SHELTER_ID')['high_pressure'].shift(-30)

df_model = df.dropna(subset=['future_pressure']).copy()

In [ ]:
# features and
X = df_model[[
    'SHELTER_ID',
    'SERVICE_USER_COUNT',
    'CAPACITY_ACTUAL_BED',
    'CAPACITY_FUNDING_BED',
    'year',
    'month',
    'dayofweek'
]]

y = df_model['future_pressure']

In [ ]:
# split the data
train = df_model[df_model['OCCUPANCY_DATE'] < '2025-01-01']
test = df_model[df_model['OCCUPANCY_DATE'] >= '2025-01-01']

X_train = train[X.columns]
y_train = train['future_pressure']

X_test = test[X.columns]
y_test = test['future_pressure']


## Cross validation for learning rate

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

In [ ]:
learning_rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4]

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
results = {}

for lr in learning_rates:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=lr,
            max_depth=6,
            max_iter=200
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        score = f1_score(y_val, preds, pos_label=1)
        fold_scores.append(score)

    results[lr] = np.mean(fold_scores)

print(results)
print("Best learning rate:", max(results, key=results.get))

KeyboardInterrupt: 

## Cross validation for max depth

In [ ]:
depths = [3, 4, 5, 6, 7, 8, 10]

tscv = TimeSeriesSplit(n_splits=5)

results = {}

for depth in depths:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=depth,
            max_iter=200
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        fold_scores.append(f1_score(y_val, preds))

    results[depth] = np.mean(fold_scores)

best_depth = max(results, key=results.get)
print(best_depth)

## Cross validation for max iteration

In [ ]:
# values to test
iters = [100, 200, 500, 1000]

tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_iter in iters:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=5,
            max_iter=n_iter
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_iter] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best max_iter:", best_iter)

{100: np.float64(0.9370125878726631), 200: np.float64(0.938097378132983), 500: np.float64(0.9385828313694443), 1000: np.float64(0.935856618094066)}
Best max_iter: 500


## Cross validation for min samples leaf

In [ ]:
leaves = [10, 20, 50, 100]
tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_leaves in leaves:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=6,
            max_iter=500,
            min_samples_leaf=n_leaves,
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_leaves] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best min_sample_leaf:", best_iter)

{10: np.float64(0.9371924753173243), 20: np.float64(0.936484846161892), 50: np.float64(0.9365819982722103), 100: np.float64(0.9359197512060071)}
Best min_sample_leaf: 10


## Cross validation for l2 reg

In [ ]:
regs = [0, 0.01, 0.1, 1.0, 10.0]
tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_regs in regs:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=6,
            max_iter=500,
            min_samples_leaf=10,
            l2_regularization=n_regs
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_regs] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best l2_regularization:", best_iter)

{0: np.float64(0.9373326757869227), 0.01: np.float64(0.9362005522412151), 0.1: np.float64(0.9361880257073565), 1.0: np.float64(0.9371847864416882), 10.0: np.float64(0.9350290513345216)}
Best l2_regularization: 0


## Cross validation for max leaf nodes

In [ ]:
nodes = [15, 31, 63, 127]
tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_nodes in nodes:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.01,
            max_depth=6,
            max_iter=500,
            min_samples_leaf=10,
            l2_regularization=0,
            max_leaf_nodes=n_nodes
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_nodes] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best max_leaf_nodes:", best_iter)

{15: np.float64(0.9357516169268157), 31: np.float64(0.9364432697487395), 63: np.float64(0.935781809044205), 127: np.float64(0.9364559304187985)}
Best max_leaf_nodes: 127


## Training the model + Results

In [ ]:
# train the model
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

train = df_model[df_model['OCCUPANCY_DATE'] < '2025-01-01']
test = df_model[df_model['OCCUPANCY_DATE'] >= '2025-01-01']

X_train = train[X.columns]
y_train = train['future_pressure']

X_test = test[X.columns]
y_test = test['future_pressure']

model = HistGradientBoostingClassifier(
    learning_rate=0.01,
    max_depth=6,
    max_iter=500,
    min_samples_leaf=10,
    l2_regularization=0,
    max_leaf_nodes=127
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

         0.0       0.90      0.25      0.39      2874
         1.0       0.89      1.00      0.94     18371

    accuracy                           0.89     21245
   macro avg       0.90      0.62      0.67     21245
weighted avg       0.89      0.89      0.87     21245



In [ ]:
# when the model says not high pressure it's right 88% of the time
  # recall: 0.25... it only catches 25% of all true “normal” cases.. it misses a lot of normal days.
  # performance of a classification model -> F1-score: 0.39 (overall weak performacne for the not high pressure class)
  # model is not good at identifying “safe / low-pressure” situations

# when the model says high pressure it's right 89% of the time
  # recall: 0.99... it catches 99% of all high cases (but look at precision, it over shoots a lot)
  # performance of a classification model -> F1-score: 0.94 ( strong performance)
  # the model is excellent at detecting shelter stress/overload

# OVERALL: this model is a very strong early warning system for shelter overload, NOT a balanced classifier for both normal and high pressure states
  # accuracy -> f1-score: 0.89 (our data is biased because we have a lot more samples of high pressure classes)
  # macro avg: taking the average performance of each class separately
  # weighted avg: takes the number of samples (support #) in each class into account

In [ ]:
# most shelter day observations in the test period were already operating above the 90% pressure threshold
# 18994/21965 ~= 86% of our test data is high pressure.

# we are saying: "given today’s conditions, we can predict with high confidence whether a shelter will be under pressure within a month."

## Dummy classifier

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

dummy_preds = dummy.predict(X_test)

print(classification_report(y_test, dummy_preds))

              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00      2874
         1.0       0.86      1.00      0.93     18371

    accuracy                           0.86     21245
   macro avg       0.43      0.50      0.46     21245
weighted avg       0.75      0.86      0.80     21245



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# To check if the model was simply exploiting class imbalance and predicting high pressure 24/7, we used
# a majority-class DummyClassifier as a baseline.
# The dummy model achieved 86% accuracy by predicting all observations as high pressure, but failed to identify normal-pressure periods (F1 = 0.00 for class 0).
#  Our HistGradientBoostingClassifier improved overall accuracy to 89% and achieved an F1-score of 0.66 compared with 0.46 for the dummy baseline
#  This shows our model learned meaningful predictive patterns beyond the majority class

## Shelter pressure Ranking (30-day outlook)

In [ ]:
# set up model probabilites
df_model['risk_prob'] = model.predict_proba(df_model[X.columns])[:, 1]

In [ ]:
# aggregate by shelter
  # avg_future_risk: overall long-term stress level
  # max_future_risk: captures spikes
  # high_risk_days: % of danger periods (0.7 -> 70% of days are high pressure)
  # total_days: how much data we have per shelter

shelter_risk = df_model.groupby('SHELTER_ID').agg(
    avg_future_risk=('risk_prob', 'mean'),
    max_future_risk=('risk_prob', 'max'),
    high_risk_days=('risk_prob', lambda x: (x > 0.8).mean()),
    total_days=('risk_prob', 'count')
).reset_index()

In [ ]:
# rank the shelters by risk
top_risk_shelters = shelter_risk.sort_values(
    'avg_future_risk',
    ascending=False
)

In [ ]:
# add location + sector info
locations = df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE', 'SECTOR']].drop_duplicates()

top_risk_shelters = top_risk_shelters.merge(
    locations,
    on='SHELTER_ID',
    how='left'
)

In [ ]:
top_risk_shelters.head(10)

,SHELTER_ID,avg_future_risk,max_future_risk,high_risk_days,total_days,LOCATION_ADDRESS,LOCATION_POSTAL_CODE,SECTOR
0,28,0.988650,0.997068,1.000000,1431,2808 Dundas St W,M6P 1Y5,Women
1,31,0.982556,0.996903,1.000000,1431,3410 Bayview Ave,M2M 3S3,Women
2,68,0.981354,0.996896,0.998602,1431,702 Kennedy Rd,M1K 2B5,Women
3,44,0.976917,0.996601,1.000000,1431,1161 Caledonia Rd,M6A 2W9,Mixed Adult
4,38,0.976743,0.997542,1.000000,1894,60 Newcastle St,M8Y 1A3,Women
5,37,0.976546,0.997542,1.000000,1809,70 Gerrard St E,M5B 1G6,Women
6,80,0.970576,0.995453,0.981348,1662,351 Lake Shore Blvd E,M5A 1C1,Mixed Adult
7,80,0.970576,0.995453,0.981348,1662,195 Princes' Blvd,M6K 3C3,Mixed Adult
8,71,0.968675,0.996668,0.994359,1241,705 Progress Ave,M1H 2X1,Mixed Adult
9,66,0.968620,0.996263,0.992596,2161,25 Augusta Ave,M5T 2K7,Mixed Adult


## Mapping shelter risks

In [56]:
!pip install geopy

In [57]:
df['FULL_ADDRESS'] = (df['LOCATION_ADDRESS'].astype(str) + ", Toronto, Canada")

In [58]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="shelter_map")

coords = {}

unique_addresses = df[['SHELTER_ID', 'FULL_ADDRESS']].drop_duplicates()

for _, row in unique_addresses.iterrows():
    address = row['FULL_ADDRESS']
    if address in coords:
        continue
    try:
        location = geolocator.geocode(address)
        coords[address] = (location.latitude, location.longitude) if location else (None, None)
    except:
        coords[address] = (None, None)
    time.sleep(1)

In [59]:
coord_df = pd.DataFrame.from_dict(coords, orient='index', columns=['lat', 'lon'])
coord_df['FULL_ADDRESS'] = coord_df.index
coord_df.reset_index(drop=True, inplace=True)

In [60]:
top_risk_shelters['FULL_ADDRESS'] = top_risk_shelters['LOCATION_ADDRESS'].astype(str) + ", Toronto, Canada"
map_df = top_risk_shelters.merge(coord_df, on='FULL_ADDRESS', how='left')
map_df = map_df.dropna(subset=['lat', 'lon'])

In [61]:
!pip install folium

In [62]:
import folium

toronto_map = folium.Map(location=[43.7, -79.4], zoom_start=11)

In [63]:
def color(r):
    if r > 0.85:
        return 'red'
    elif r > 0.7:
        return 'orange'
    else:
        return 'green'

In [64]:
for _, row in map_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color(row['avg_future_risk']),
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['LOCATION_ADDRESS']} | Risk: {row['avg_future_risk']:.2f}"
    ).add_to(toronto_map)

toronto_map

## Export coordinates + pressure risk level

In [54]:
def pressure_label(r):
    if r > 0.85:
        return 'High'
    elif r > 0.7:
        return 'Medium'
    else:
        return 'Low'

export_df = map_df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE',
                     'lat', 'lon', 'avg_future_risk']].copy()
export_df = export_df.drop_duplicates(subset=['SHELTER_ID', 'LOCATION_ADDRESS'])
export_df['pressure_risk'] = export_df['avg_future_risk'].apply(pressure_label)
export_df = export_df.rename(columns={'lat': 'LATITUDE', 'lon': 'LONGITUDE'})

export_df.to_csv(folder + 'bed_based_shelters_map.csv', index=False)
print(f"Saved {len(export_df)} shelters")
print(export_df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LATITUDE', 'LONGITUDE', 'pressure_risk']].to_string(index=False))

Saved 68 shelters
 SHELTER_ID               LOCATION_ADDRESS  LATITUDE  LONGITUDE pressure_risk
         28               2808 Dundas St W 43.665538 -79.463127          High
         31               3410 Bayview Ave 43.798251 -79.395423          High
         68                 702 Kennedy Rd 43.728050 -79.267710          High
         44              1161 Caledonia Rd 43.715628 -79.467412          High
         38                60 Newcastle St 43.617951 -79.497359          High
         37                70 Gerrard St E 43.660096 -79.378352          High
         80          351 Lake Shore Blvd E 43.647838 -79.359354          High
         80              195 Princes' Blvd 43.632090 -79.420900          High
         71               705 Progress Ave 43.776710 -79.246020          High
         66                 25 Augusta Ave 43.648993 -79.399668          High
         66                  69 Fraser Ave 43.637870 -79.423660          High
         77             2671 Islington Ave 43.

In [55]:
from google.colab import files
files.download(folder + 'bed_based_shelters_map.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>